In [8]:
import boto3
import AWS_Connection
import numpy as np
import pandas as pd
array_of_files = np.array(AWS_Connection.getListRoutesFiles(contiene='.csv.gz', evitar='routes'))
unique_registries = np.unique([file for file in array_of_files])
array_of_files

array(['cityflow/mx/hidalgo_state_mx/2023-06/cityflow/citydata_hidalgo_state_mx_2023-06_deviceTrips_cityflow_00000.csv.gz',
       'cityflow/mx/hidalgo_state_mx/2023-10/cityflow/citydata_hidalgo_state_mx_2023-10_deviceTrips_cityflow_00000.csv.gz',
       'cityflow/mx/hidalgo_state_mx/2023-10/cityflow/citydata_hidalgo_state_mx_2023-10_deviceTrips_cityflow_00001.csv.gz',
       'cityflow/mx/hidalgo_state_mx/2023-12/cityflow/citydata_hidalgo_state_mx_2023-12_deviceTrips_cityflow_00000.csv.gz',
       'cityflow/mx/hidalgo_state_mx/2023-12/cityflow/citydata_hidalgo_state_mx_2023-12_deviceTrips_cityflow_00001.csv.gz'],
      dtype='<U113')

In [9]:
import boto3
from io import TextIOWrapper
from gzip import GzipFile
import csv
import pandas as pd
import os
from AWS_Credentials import access_key, secret_key
import geopandas as gpd
from shapely.geometry import Point

archivo_shp = gpd.read_file('Datos/Tuzobus/Troncal/Estaciones shp/estaciones_buffer_25mts.shp')
archivo_shp.crs

def getDataGivenIndex(file,desired_columns=None, max_rows=100):
    s3 = boto3.client('s3', aws_access_key_id=access_key,
        aws_secret_access_key=secret_key)
    response = s3.get_object(Bucket='citydata.hidalgo', Key=file)
    gzipped = GzipFile(None, 'rb', fileobj=response['Body'])
    #print(gzipped)
    data = TextIOWrapper(gzipped)
    #print(data)
    # Initialize counters
    total_lines = 0
    selected_lines = 0

    # Desired columns

    # Create a CSV reader
    reader = csv.DictReader(data)

    # Create an empty DataFrame with desired columns

    # Process the data
    for row in reader:
        if(total_lines==0):
            if desired_columns == None:
                desired_columns = row.keys()
                df_final = pd.DataFrame(columns=row.keys())
            else:
                df_final = pd.DataFrame(columns=desired_columns)
                
        total_lines += 1
        # selected_lines += 1
        longitud = float(row['overlap_destination_long'])
        latitud = float(row['overlap_destination_lat'])
        punto = Point(longitud, latitud)
        if archivo_shp.contains(punto).any():
            df_final.loc[selected_lines] = [row[col] for col in desired_columns]
            selected_lines += 1
            # print(selected_lines)
            if selected_lines >= max_rows:
                break
    print(total_lines,selected_lines)
    return(df_final)

In [10]:
for i in range(0, len(array_of_files)):
    print("Vamos en el archivo: ", i, " de ", len(array_of_files))
    datos = getDataGivenIndex(array_of_files[i], max_rows= np.inf)
    nombre_array = str(array_of_files[i])  # Asegúrate de que sea string
    nombre = nombre_array.split('/')[-1]  # citydata_hidalgo_state_mx_2023-06_deviceTrips_cityflow_00000.csv.gz
    nombre = nombre.replace('.csv.gz', '')  # Elimina la extensión
    datos.to_csv('Datos/Tuzobus/Troncal/Citydata_filtracion/' + nombre + '.csv', index=False)

Vamos en el archivo:  0  de  5
4109903 3729
Vamos en el archivo:  1  de  5
5000000 4386
Vamos en el archivo:  2  de  5
4569245 3897
Vamos en el archivo:  3  de  5
5000000 3862
Vamos en el archivo:  4  de  5
4179308 3596
